In [1]:
from sympy import Array, Symbol, tensorcontraction, tensorproduct
import black


def bending_function_sympy(Q, Bp, Bpp):
    QM = Q.reshape(4, 3).transpose()

    # Projections (3-vector each)

    yip = tensorcontraction(tensorproduct(QM, Bp), (1, 2))
    yipp = tensorcontraction(tensorproduct(QM, Bpp), (1, 2))

    def cross(a: Array, b: Array):
        """
        3d cross product
        https://reference.wolfram.com/language/ref/Cross.html
        """
        return Array([
            a[1] * b[2] - a[2] * b[1],
            a[2] * b[0] - a[0] * b[2],
            a[0] * b[1] - a[1] * b[0],
        ])

    def norm3(arr: Array):
        x, y, z = arr[0], arr[1], arr[2]
        return (x**2 + y**2 + z**2) ** (0.5)

    return (norm3(cross(yip, yipp)) / (norm3(yip) ** 3)) ** 2


sympy_res = bending_function_sympy(
    Array([Symbol(f"q{i}") for i in range(1, 13)]),
    Array([Symbol(f"bp{i}") for i in range(1, 5)]),
    Array([Symbol(f"bpp{i}") for i in range(1, 5)]),
)
print(black.format_str(repr(sympy_res), mode=black.Mode(line_length=88)).strip())

(
    (
        (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
        - (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        -(bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q1 + bpp2 * q4 + bpp3 * q7 + bpp4 * q10)
    )
    ** 2
    + (
        (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11)
        * (bpp1 * q3 + bpp2 * q6 + bpp3 * q9 + bpp4 * q12)
        - (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12)
        * (bpp1 * q2 + bpp2 * q5 + bpp3 * q8 + bpp4 * q11)
    )
    ** 2
) ** 1.0 / (
    (bp1 * q1 + bp2 * q4 + bp3 * q7 + bp4 * q10) ** 2
    + (bp1 * q2 + bp2 * q5 + bp3 * q8 + bp4 * q11) ** 2
    + (bp1 * q3 + bp2 * q6 + bp3 * q9 + bp4 * q12) ** 2
) ** 3.0


In [ ]:
import numpy as np

# create random array inputs
ng = np.random.default_rng()
# control points
q_np = ng.random(12).astype(float)
# bernstein coefficients
bp_np = ng.random(4).astype(float)
bpp_np = ng.random(4).astype(float)
# applicaiton layer
# polynomial layer
# - next step do deriviates from mathematica
# egraph layer

In [3]:
def bending_function(Q, Bp, Bpp):
    xp = Q.__array_namespace__()
    QM = xp.reshape(Q, (4, 3)).T

    yip = xp.vecdot(QM, Bp)
    yipp = xp.vecdot(QM, Bpp)
    num = xp.linalg.vector_norm(xp.cross(yip, yipp))
    den = xp.linalg.vector_norm(yip) ** 3
    return (num / den) ** 2


bending_function(q_np, bp_np, bpp_np)

# Try sympy on random inputs to make sure they agree
(
    bending_function_sympy(
        Array(q_np.tolist()),
        Array(bp_np.tolist()),
        Array(bpp_np.tolist()),
    ),
    bending_function(q_np, bp_np, bpp_np),
)

(0.00303248851892518, np.float64(0.0030324885189251874))

In [4]:
import egglog
import egglog.exp.array_api as enp

Bp = enp.NDArray.vector([egglog.constant(f"bp{i}", enp.Value) for i in range(1, 5)])
Bpp = enp.NDArray.vector([egglog.constant(f"bpp{i}", enp.Value) for i in range(1, 5)])
Q = enp.NDArray.vector([egglog.constant(f"q{i}", enp.Value) for i in range(1, 13)])
res = bending_function(Q, Bp, Bpp)
res

_NDArray_1 = reshape(
    NDArray.vector(
        TupleValue.from_vec(
            Vec[Value](q1, q2, q3, q4, q5, q6, q7, q8, q9, q10, q11, q12)
        )
    ),
    TupleInt.from_vec(Vec[Int](Int(4), Int(3))),
).T
_NDArray_2 = vecdot(
    _NDArray_1, NDArray.vector(TupleValue.from_vec(Vec[Value](bp1, bp2, bp3, bp4)))
)
(
    vector_norm(
        cross(
            _NDArray_2,
            vecdot(
                _NDArray_1,
                NDArray.vector(TupleValue.from_vec(Vec[Value](bpp1, bpp2, bpp3, bpp4))),
            ),
        )
    )
    / vector_norm(_NDArray_2) ** NDArray.scalar(Value.int(Int(3)))
) ** NDArray.scalar(Value.int(Int(2)))

In [5]:
# egraph = egglog.EGraph()
# egraph.register(res)
# egraph.run(enp.array_api_combined_ruleset.saturate())
# egraph.extract(res)
res_simp = res.eval()
res_simp

_Value_1 = q2 * bp1 + q5 * bp2 + q8 * bp3 + q11 * bp4
_Value_2 = q3 * bpp1 + q6 * bpp2 + q9 * bpp3 + q12 * bpp4
_Value_3 = q3 * bp1 + q6 * bp2 + q9 * bp3 + q12 * bp4
_Value_4 = q2 * bpp1 + q5 * bpp2 + q8 * bpp3 + q11 * bpp4
_Value_5 = q1 * bpp1 + q4 * bpp2 + q7 * bpp3 + q10 * bpp4
_Value_6 = q1 * bp1 + q4 * bp2 + q7 * bp3 + q10 * bp4
(
    (_Value_1 * _Value_2 - _Value_3 * _Value_4) ** Value.int(Int(2))
    + (_Value_3 * _Value_5 - _Value_6 * _Value_2) ** Value.int(Int(2))
    + (_Value_6 * _Value_4 - _Value_1 * _Value_5) ** Value.int(Int(2))
) / (
    _Value_6 ** Value.int(Int(2))
    + _Value_1 ** Value.int(Int(2))
    + _Value_3 ** Value.int(Int(2))
) ** Value.int(
    Int(3)
)

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from functools import total_ordering


@total_ordering
@dataclass(frozen=True)
class ArithmeticCost:
    """
    Cost type where *, +, and ** are the same and then / is more than all of them.
    """

    muls: int = 0
    divs: int = 0
    adds: int = 0
    subs: int = 0
    exps: int = 0

    @property
    def not_div_ops(self) -> int:
        return self.muls + self.adds + self.exps + self.subs

    def __eq__(self, other: ArithmeticCost) -> bool:
        return self.not_div_ops == other.not_div_ops and self.divs == other.divs

    def __gt__(self, other: ArithmeticCost) -> bool:
        if self.divs != other.divs:
            return self.divs > other.divs
        return self.not_div_ops > other.not_div_ops

    def __add__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls + other.muls,
            divs=self.divs + other.divs,
            adds=self.adds + other.adds,
            exps=self.exps + other.exps,
            subs=self.subs + other.subs,
        )

    def __sub__(self, other: ArithmeticCost) -> ArithmeticCost:
        return ArithmeticCost(
            muls=self.muls - other.muls,
            divs=self.divs - other.divs,
            adds=self.adds - other.adds,
            exps=self.exps - other.exps,
            subs=self.subs - other.subs,
        )


@egglog.greedy_dag_cost_model
def arith_cost_model(egraph: egglog.EGraph, expr: egglog.Expr, children_costs: list[ArithmeticCost]) -> ArithmeticCost:
    # named variables are free
    if egglog.get_constant_name(expr) is not None:
        return ArithmeticCost()
    # literals are free
    if egglog.get_literal_value(expr) is not None:
        return ArithmeticCost()
    match egglog.get_callable_fn(expr):
        case enp.Value.__add__:
            c = ArithmeticCost(adds=1)
        case enp.Value.__sub__:
            c = ArithmeticCost(subs=1)
        case enp.Value.__mul__:
            c = ArithmeticCost(muls=1)
        case enp.Value.__truediv__:
            c = ArithmeticCost(divs=1)
        case enp.Value.__pow__:
            c = ArithmeticCost(exps=1)
        case enp.Int | enp.Value.int:
            c = ArithmeticCost()
        case _ as fn:
            raise NotImplementedError(f"Unknown expr for arith cost model: {fn or expr}")
    return sum(children_costs, c)


egraph = egglog.EGraph()
egraph.register(res_simp)
cost = egraph.extract(res_simp, cost_model=arith_cost_model, include_cost=True)[1]
print(cost.total.not_div_ops)
cost

62


GreedyDagCost(total=ArithmeticCost(muls=30, divs=1, adds=22, subs=3, exps=7))

Now let's try distributing...

In [7]:
@egglog.ruleset
def distribute(a: enp.Value, b: enp.Value, c: enp.Value):
    yield egglog.rewrite((a + b) * c, subsume=True).to(a * c + b * c)
    yield egglog.rewrite(c * (a + b), subsume=True).to(c * a + c * b)
    yield egglog.rewrite((a - b) * c, subsume=True).to(a * c - b * c)
    yield egglog.rewrite(c * (a - b), subsume=True).to(c * a - c * b)
    # Push down subtraction
    yield egglog.rewrite(a - (b + c), subsume=True).to(a - b - c)
    yield egglog.rewrite(a - (b - c), subsume=True).to(a - b + c)


egraph.run(distribute.saturate())
res, cost = egraph.extract(res_simp, cost_model=arith_cost_model, include_cost=True)
print(cost.total.not_div_ops)
print(cost)
res

233
GreedyDagCost(total=ArithmeticCost(muls=120, divs=1, adds=58, subs=48, exps=7))


(
    (
        q2 * bp1 * (q3 * bpp1)
        + q2 * bp1 * (q6 * bpp2)
        + q2 * bp1 * (q9 * bpp3)
        + q2 * bp1 * (q12 * bpp4)
        + (
            q5 * bp2 * (q3 * bpp1)
            + q5 * bp2 * (q6 * bpp2)
            + q5 * bp2 * (q9 * bpp3)
            + q5 * bp2 * (q12 * bpp4)
        )
        + (
            q8 * bp3 * (q3 * bpp1)
            + q8 * bp3 * (q6 * bpp2)
            + q8 * bp3 * (q9 * bpp3)
            + q8 * bp3 * (q12 * bpp4)
        )
        + (
            q11 * bp4 * (q3 * bpp1)
            + q11 * bp4 * (q6 * bpp2)
            + q11 * bp4 * (q9 * bpp3)
            + q11 * bp4 * (q12 * bpp4)
        )
        - q3 * bp1 * (q2 * bpp1)
        - q6 * bp2 * (q2 * bpp1)
        - q3 * bp1 * (q5 * bpp2)
        - q6 * bp2 * (q5 * bpp2)
        - q3 * bp1 * (q8 * bpp3)
        - q6 * bp2 * (q8 * bpp3)
        - q3 * bp1 * (q11 * bpp4)
        - q6 * bp2 * (q11 * bpp4)
        - q9 * bp3 * (q2 * bpp1)
        - q9 * bp3 * (q5 * bpp2)
        - q9 * bp3 * (q8 * bpp3)
        - q9 * bp3 * (q11 * bpp4)
        - q12 * bp4 * (q2 * bpp1)
        - q12 * bp4 * (q5 * bpp2)
        - q12 * bp4 * (q8 * bpp3)
        - q12 * bp4 * (q11 * bpp4)
    )
    ** Value.int(Int(2))
    + (
        q3 * bp1 * (q1 * bpp1)
        + q3 * bp1 * (q4 * bpp2)
        + q3 * bp1 * (q7 * bpp3)
        + q3 * bp1 * (q10 * bpp4)
        + (
            q6 * bp2 * (q1 * bpp1)
            + q6 * bp2 * (q4 * bpp2)
            + q6 * bp2 * (q7 * bpp3)
            + q6 * bp2 * (q10 * bpp4)
        )
        + (
            q9 * bp3 * (q1 * bpp1)
            + q9 * bp3 * (q4 * bpp2)
            + q9 * bp3 * (q7 * bpp3)
            + q9 * bp3 * (q10 * bpp4)
        )
        + (
            q12 * bp4 * (q1 * bpp1)
            + q12 * bp4 * (q4 * bpp2)
            + q12 * bp4 * (q7 * bpp3)
            + q12 * bp4 * (q10 * bpp4)
        )
        - q1 * bp1 * (q3 * bpp1)
        - q4 * bp2 * (q3 * bpp1)
        - q1 * bp1 * (q6 * bpp2)
        - q4 * bp2 * (q6 * bpp2)
        - q1 * bp1 * (q9 * bpp3)
        - q4 * bp2 * (q9 * bpp3)
        - q1 * bp1 * (q12 * bpp4)
        - q4 * bp2 * (q12 * bpp4)
        - q7 * bp3 * (q3 * bpp1)
        - q7 * bp3 * (q6 * bpp2)
        - q7 * bp3 * (q9 * bpp3)
        - q7 * bp3 * (q12 * bpp4)
        - q10 * bp4 * (q3 * bpp1)
        - q10 * bp4 * (q6 * bpp2)
        - q10 * bp4 * (q9 * bpp3)
        - q10 * bp4 * (q12 * bpp4)
    )
    ** Value.int(Int(2))
    + (
        q1 * bp1 * (q2 * bpp1)
        + q1 * bp1 * (q5 * bpp2)
        + q1 * bp1 * (q8 * bpp3)
        + q1 * bp1 * (q11 * bpp4)
        + (
            q4 * bp2 * (q2 * bpp1)
            + q4 * bp2 * (q5 * bpp2)
            + q4 * bp2 * (q8 * bpp3)
            + q4 * bp2 * (q11 * bpp4)
        )
        + (
            q7 * bp3 * (q2 * bpp1)
            + q7 * bp3 * (q5 * bpp2)
            + q7 * bp3 * (q8 * bpp3)
            + q7 * bp3 * (q11 * bpp4)
        )
        + (
            q10 * bp4 * (q2 * bpp1)
            + q10 * bp4 * (q5 * bpp2)
            + q10 * bp4 * (q8 * bpp3)
            + q10 * bp4 * (q11 * bpp4)
        )
        - q2 * bp1 * (q1 * bpp1)
        - q5 * bp2 * (q1 * bpp1)
        - q2 * bp1 * (q4 * bpp2)
        - q5 * bp2 * (q4 * bpp2)
        - q2 * bp1 * (q7 * bpp3)
        - q5 * bp2 * (q7 * bpp3)
        - q2 * bp1 * (q10 * bpp4)
        - q5 * bp2 * (q10 * bpp4)
        - q8 * bp3 * (q1 * bpp1)
        - q8 * bp3 * (q4 * bpp2)
        - q8 * bp3 * (q7 * bpp3)
        - q8 * bp3 * (q10 * bpp4)
        - q11 * bp4 * (q1 * bpp1)
        - q11 * bp4 * (q4 * bpp2)
        - q11 * bp4 * (q7 * bpp3)
        - q11 * bp4 * (q10 * bpp4)
    )
    ** Value.int(Int(2))
) / (
    (q1 * bp1 + q4 * bp2 + q7 * bp3 + q10 * bp4) ** Value.int(Int(2))
    + (q2 * bp1 + q5 * bp2 + q8 * bp3 + q11 * bp4) ** Value.int(Int(2))
    + (q3 * bp1 + q6 * bp2 + q9 * bp3 + q12 * bp4) ** Value.int(Int(2))
) ** Value.int(
    Int(3)
)

In [8]:
egraph = egglog.EGraph()
egraph.register(res_simp)
egraph.run(enp.polynomial_schedule)
egraph.extract(res_simp)

_Value_1 = polynomial(
    MultiSet(
        MultiSet(q2, bp1), MultiSet(bp2, q5), MultiSet(bp3, q8), MultiSet(q11, bp4)
    )
)
_Value_2 = polynomial(
    MultiSet(
        MultiSet(bpp1, q3), MultiSet(q6, bpp2), MultiSet(q9, bpp3), MultiSet(q12, bpp4)
    )
)
_Value_3 = polynomial(
    MultiSet(
        MultiSet(bp1, q3), MultiSet(bp2, q6), MultiSet(q9, bp3), MultiSet(q12, bp4)
    )
)
_Value_4 = polynomial(
    MultiSet(
        MultiSet(bpp1, q2), MultiSet(q5, bpp2), MultiSet(q8, bpp3), MultiSet(bpp4, q11)
    )
)
_Value_5 = polynomial(
    MultiSet(
        MultiSet(bpp1, q1), MultiSet(bpp2, q4), MultiSet(q7, bpp3), MultiSet(bpp4, q10)
    )
)
_Value_6 = polynomial(
    MultiSet(
        MultiSet(bp1, q1), MultiSet(bp2, q4), MultiSet(q7, bp3), MultiSet(q10, bp4)
    )
)
polynomial(
    MultiSet(
        MultiSet(
            polynomial(
                MultiSet(
                    MultiSet(_Value_1, _Value_2),
                    MultiSet(_Value_3, _Value_4, Value.int(Int(-1))),
                )
            )
            ** Value.int(Int(2))
        ),
        MultiSet(
            polynomial(
                MultiSet(
                    MultiSet(_Value_3, _Value_5),
                    MultiSet(_Value_6, _Value_2, Value.int(Int(-1))),
                )
            )
            ** Value.int(Int(2))
        ),
        MultiSet(
            polynomial(
                MultiSet(
                    MultiSet(_Value_4, _Value_6),
                    MultiSet(_Value_1, _Value_5, Value.int(Int(-1))),
                )
            )
            ** Value.int(Int(2))
        ),
    )
) / polynomial(
    MultiSet(
        MultiSet(_Value_6 ** Value.int(Int(2))),
        MultiSet(_Value_1 ** Value.int(Int(2))),
        MultiSet(_Value_3 ** Value.int(Int(2))),
    )
) ** Value.int(
    Int(3)
)

In [ ]:
# CSE while doing polynomial to bin ops, i.e. a*c vs c*a

# enhance with bernstein coefficients and derivatives